# IPCC LULUCF zonal stats summary spreadsheet by country (adm0)

Reads the outputs of `0_calculate_LULUCF_vegetation_soil_IPCC_zonal_stats.py` (or `combine_IPCC_LULUCF_zonal_stats.py`) for one run and writes one Excel workbook with two tabs:

- **flux**: `value` and `area_ha` summed by country, flux source, gas, IPCC classes, primary forest, year and analysis layer. `area_ha` is the area of rows with non-zero flux for that layer, not total land area.
- **ipcc_class_area**: total area (all pixels, whether or not they have flux) by country, IPCC class, change and summary (codes and names) and year. IPCC node code is summed out of both tabs. Use this for total country/class areas.

Before the tabs are summed, the **IPCC class reassignment rules** from `1_global_LULUCF_vegetation_IPCC_summary_spreadsheet.ipynb` are applied to both: rows whose `IPCC_class_name` is `Unassigned` get an IPCC class, change and summary from their vegetation model land state (`land_state_broad_class`, `tall_veg_type`). `IPCC_change_name` and `IPCC_summary_name` are updated to match the new codes. Land state and primary forest columns are then summed out of the class area tab.

`IPCC_initial` is the first digit of `IPCC_change` (the IPCC class at the start of the change interval; 0 for Unassigned). It is calculated after the reassignment rules and is in both tabs.

`flux_source` is taken from the analysis layer name prefix: `LULUCF_`, `veg_`, `SOC_` (mineral soil) or `organic_soil_` (also `burned_`/`drained_` in older outputs).

The newest combined files matching `zonal_stats_description` are used. If the run was not combined, the per-tile files are read instead: flux tiles (`ipcc_lulucf_zonal_stats_<tile>_…`) from `flux_dir` and IPCC class area tiles (`ipcc_class_area_<tile>_…`) from `class_area_dir`, summed across tiles here. For each tile only the newest file is used. Set `repo_root` below so `src.utilities` can be imported.

In [ ]:
from pathlib import Path
import gc
import importlib.util
import re
import sys

import pandas as pd

# Root of the AFOLU flux model repo for the shared utilities import.
repo_root = Path(r"/mnt/c/GIS/git/AFOLU_GHG_flux_model")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

### User Inputs

In [ ]:
# Folders with the zonal stats outputs for the run. Combined files (ipcc_lulucf_zonal_stats_v..., ipcc_class_area_by_country_v...)
# are written to flux_dir; per-tile class area files are in class_area_dir.
flux_dir = Path(r"/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global")
class_area_dir = flux_dir / "ipcc_class_area"

# -zd used for the run; only files with this description are used
zonal_stats_description = "EU27"

# Keep only the EU27 countries
FILTER_TO_EU27 = True

EU27_ISO3 = [
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL",
    "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE",
]

# Countries to keep when FILTER_TO_EU27 is False (ISO3). Set to None to keep every country.
other_countries = None

countries = EU27_ISO3 if FILTER_TO_EU27 else other_countries

# Workbook is written to flux_dir
output_dir = flux_dir

print("Flux folder:", flux_dir, "| exists:", flux_dir.exists())
print("IPCC class area folder:", class_area_dir, "| exists:", class_area_dir.exists())
print("Zonal stats description:", zonal_stats_description)
print("Countries:", "EU27" if FILTER_TO_EU27 else (countries if countries else "all"))

### IPCC class reassignment rules

Same rules as `1_global_LULUCF_vegetation_IPCC_summary_spreadsheet.ipynb`. A rule only applies to rows whose `IPCC_class_name` is `Unassigned`, and fills the IPCC columns in `fill` when every column in `match` has the given value.

In [ ]:
numeric_to_ipcc_class = {
    0: "Unassigned",
    1: "Settlements",
    2: "Cropland",
    3: "Forest Land",
    4: "Grassland",
    5: "Wetlands",
    6: "Other Land (Bare)",
    7: "Other Land (Water)",
    8: "Other Land (Snow/Ice)",
}

assignment_rules = [
    {
        "name": "crop_to_cropland",
        "match": {"land_state_broad_class": "crop"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "short_veg_to_grassland",
        "match": {"land_state_broad_class": "short_veg"},
        "fill": {"IPCC_summary": 44, "IPCC_change": 44, "IPCC_class": 4, "IPCC_class_name": "Grassland"},
    },
    {
        "name": "mangrove_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "mangrove"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "natural_tree_cover_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "natural_tree_cover"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "oil_palm_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "oil_palm"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "non_oil_palm_planted_trees_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "non_oil_palm_planted_trees"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "trees_in_other_land_covers_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "trees_in_other_land_covers"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "no_flux_stays_unassigned",
        "match": {"land_state_broad_class": "no_flux"},
        "fill": {"IPCC_summary": 0, "IPCC_change": 0, "IPCC_class": 0, "IPCC_class_name": "Unassigned"},
    },
]

In [ ]:
# Flux tab: columns summed over (only those present in the run are used)
FLUX_GROUP_COLS = [
    cn.adm0_pattern,
    "country_name",
    "flux_source",
    "gas",
    "IPCC_summary",
    "IPCC_summary_name",
    #"IPCC_node_code",
    #"IPCC_node_code_name",
    "year",
    "IPCC_change",
    "IPCC_change_name",
    "IPCC_initial",
    "IPCC_class",
    "IPCC_class_name",
    "starting_composite_primary_forest",
    "analysis_layer",
]
VALUE_COLS = ["value", "area_ha"]

# ipcc_class_area tab: IPCC class, change and summary columns from the class area parquets, plus IPCC_initial.
# IPCC node code, region, primary forest, land state and tile columns are summed out.
CLASS_AREA_GROUP_COLS = [
    cn.adm0_pattern, "country_name",
    "IPCC_class", "IPCC_class_name",
    "IPCC_change", "IPCC_change_name",
    "IPCC_initial",
    "IPCC_summary", "IPCC_summary_name",
    #"IPCC_node_code", "IPCC_node_code_name",
    "year",
]
CLASS_AREA_COLS = CLASS_AREA_GROUP_COLS + ["area_ha"]

# Land state columns the reassignment rules match on
RULE_MATCH_COLS = sorted({col for rule in assignment_rules for col in rule["match"]})

# Analysis layer name prefix -> flux source
FLUX_SOURCE_PREFIXES = {
    "LULUCF_": "LULUCF",
    "veg_": "vegetation",
    "SOC_": "mineral_soil",
    "organic_soil_": "organic_soil",
    "burned_": "organic_soil",     # Organic soil layer names in outputs made before the organic_soil_ prefix
    "drained_": "organic_soil",
}

EXCEL_MAX_ROWS = 1_048_575   # Excel's row limit, minus the header row


def log(message):
    print(f"{uu.timestr()}: {message}", flush=True)


TILE_RE = re.compile(r"^(?P<prefix>.+?)_(?P<tile>\d{2}[NS]_\d{3}[EW])_")


def _matches_run(p, prefix, exclude):
    """File has this prefix, exactly this run's description followed by the output timestamp, and none of the excluded
    substrings. Exact, so 'global' doesn't also match files from a run described as e.g. 'global_test'."""
    name = p.name
    desc = re.escape(zonal_stats_description)
    timestamp = r"\d{8}_\d{2}_\d{2}_\d{2}\.parquet$"
    return (name.startswith(prefix)
            and (re.search(rf"_{desc}_{timestamp}", name) is not None
                 or re.match(rf"{re.escape(prefix)}{desc}_{timestamp}", name) is not None)
            and not any(e in name for e in exclude))


def newest_file(folder, prefix, exclude=()):
    """Newest combined parquet for the run, or None if there isn't one (tile files and other descriptions are skipped)."""
    candidates = [p for p in Path(folder).glob(f"{prefix}*.parquet")
                  if _matches_run(p, prefix, exclude) and not TILE_RE.match(p.name)]
    if not candidates:
        return None
    newest = max(candidates, key=lambda p: p.stat().st_mtime)
    log(f"Using {newest.name}" + (f" (newest of {len(candidates)})" if len(candidates) > 1 else ""))
    return newest


def newest_tile_files(folder, prefix, exclude=()):
    """Newest per-tile parquet for each 10x10 tile of the run."""
    by_tile = {}
    for p in Path(folder).glob(f"{prefix}*.parquet"):
        m = TILE_RE.match(p.name)
        if m and _matches_run(p, prefix, exclude):
            tile = m.group("tile")
            if tile not in by_tile or p.stat().st_mtime > by_tile[tile].stat().st_mtime:
                by_tile[tile] = p
    return [by_tile[t] for t in sorted(by_tile)]


def read_run(folders, combined_prefix, tile_prefix, exclude=()):
    """Reads the combined file for the run if one exists in any of the folders, otherwise concatenates the tile files.
    Returns (dataframe, description of what was read)."""
    folders = [Path(f) for f in folders if Path(f).is_dir()]
    for folder in folders:
        combined = newest_file(folder, combined_prefix, exclude)
        if combined is not None:
            return pd.read_parquet(combined), combined.name
    for folder in folders:
        tiles = newest_tile_files(folder, tile_prefix, exclude)
        if tiles:
            log(f"No combined {combined_prefix}* file; reading {len(tiles)} tile files from {folder}:")
            for t in tiles:
                log(f"    {t.name}")
            return pd.concat([pd.read_parquet(t) for t in tiles], ignore_index=True), f"{len(tiles)} tiles in {folder}"
    raise FileNotFoundError(f"No {combined_prefix}* or {tile_prefix}* parquet for '{zonal_stats_description}' in {[str(f) for f in folders]}")


def flux_source(layer):
    for prefix, source in FLUX_SOURCE_PREFIXES.items():
        if layer.startswith(prefix):
            return source
    return "other"


def keep_countries(df, table_name):
    if countries:
        df = df[df[cn.adm0_pattern].isin(countries)].copy()
        missing = sorted(set(countries) - set(df[cn.adm0_pattern].unique()))
        if missing:
            log(f"WARNING: {table_name} has no rows for {missing}")
    return df


def apply_assignment_rule(df, table_name, rules=assignment_rules):
    """IPCC class reassignment from the global summary notebook: fills IPCC class, change and summary for Unassigned rows
    from their land state. Also updates IPCC_change_name and IPCC_summary_name for the rows it changes.
    Returns the df and a report of rows (and area) changed by each rule."""
    missing = [c for c in RULE_MATCH_COLS + ["IPCC_class_name"] if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name} has no {missing} columns, so the IPCC class reassignment rules can't be applied")

    df["IPCC_class_name"] = df["IPCC_class_name"].fillna("Unassigned")
    base_unassigned = df["IPCC_class_name"].astype(str).eq("Unassigned")
    reassigned = pd.Series(False, index=df.index)
    report = []

    for rule in rules:
        mask = base_unassigned.copy()

        for col, val in rule["match"].items():
            mask &= df[col].astype(str).eq(str(val))

        if mask.any():
            for fill_col, fill_val in rule["fill"].items():
                df.loc[mask, fill_col] = fill_val
            reassigned |= mask

        report.append({"table": table_name, "rule": rule["name"], "rows": int(mask.sum()),
                       "area_ha": float(df.loc[mask, "area_ha"].sum())})

    for col in ["IPCC_summary", "IPCC_change", "IPCC_class"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

    class_map_mask = df["IPCC_class_name"].isna() | df["IPCC_class_name"].astype(str).eq("")
    if "IPCC_class" in df.columns:
        df.loc[class_map_mask, "IPCC_class_name"] = df.loc[class_map_mask, "IPCC_class"].map(numeric_to_ipcc_class).fillna("Unassigned")

    # Change and summary names that match the reassigned codes
    for col in ["IPCC_change", "IPCC_summary"]:
        if f"{col}_name" in df.columns:
            df.loc[reassigned, f"{col}_name"] = df.loc[reassigned, col].map(cn.numeric_to_ipcc_change).fillna("Unassigned")

    log(f"{table_name}: reassignment rules changed {int(reassigned.sum()):,} of {int(base_unassigned.sum()):,} Unassigned rows")
    return df, pd.DataFrame(report)


def add_ipcc_initial(df):
    """IPCC_initial: first digit of IPCC_change (codes are 10 x initial class + final class; 0 is Unassigned)."""
    df["IPCC_initial"] = (df["IPCC_change"].astype("int64") // 10).astype("int64")
    return df


def summarize_flux(df, source):
    df = keep_countries(df, "flux")
    log(f"Flux: {len(df):,} input rows for {sorted(df[cn.adm0_pattern].unique())}")
    if df.empty:
        raise ValueError(f"No flux rows for {countries} in {source}")

    df["flux_source"] = df["analysis_layer"].map(flux_source)
    unknown = sorted(df.loc[df["flux_source"] == "other", "analysis_layer"].unique())
    if unknown:
        log(f"WARNING: no flux source for {unknown}; labelled 'other'")

    # Values are float32 in the parquet; sum in float64 so totals don't lose precision
    df[VALUE_COLS] = df[VALUE_COLS].astype("float64")
    input_totals = df.groupby("analysis_layer")[VALUE_COLS].sum()

    df, rule_report = apply_assignment_rule(df, "flux")
    df = add_ipcc_initial(df)

    group_cols = [c for c in FLUX_GROUP_COLS if c in df.columns]
    summary = (
        df.groupby(group_cols, dropna=False, observed=True, as_index=False)[VALUE_COLS]
          .sum()
          .sort_values(group_cols)
          .reset_index(drop=True)
    )
    del df
    gc.collect()

    # Reassigning and dropping detail columns only moves and merges rows, so totals per layer must be unchanged
    output_totals = summary.groupby("analysis_layer")[VALUE_COLS].sum()
    diff = (output_totals - input_totals).abs().max().max()
    log(f"Flux: {len(summary):,} output rows; max total difference vs input = {diff:,.6f}")

    return summary, rule_report


def summarize_class_area(df, source):
    df = keep_countries(df, "IPCC class area")
    log(f"IPCC class area: {len(df):,} input rows for {sorted(df[cn.adm0_pattern].unique())}")
    if df.empty:
        raise ValueError(f"No IPCC class area rows for {countries} in {source}")

    needed = [c for c in CLASS_AREA_COLS + RULE_MATCH_COLS if c not in df.columns and c != "IPCC_initial"]
    if needed:
        raise ValueError(f"IPCC class area data ({source}) has no {needed} columns. It needs to come from the zonal stats "
                         f"script version that writes IPCC change/summary/node and land state to the class area tables.")

    df["area_ha"] = df["area_ha"].astype("float64")
    input_totals = df.groupby([cn.adm0_pattern, "year"])["area_ha"].sum()

    df, rule_report = apply_assignment_rule(df, "ipcc_class_area")
    df = add_ipcc_initial(df)

    # Sorted by whichever of these columns are in CLASS_AREA_GROUP_COLS (so columns can be commented out of it)
    sort_cols = [c for c in [cn.adm0_pattern, "year", "IPCC_class", "IPCC_change", "IPCC_summary", "IPCC_node_code"]
                 if c in CLASS_AREA_GROUP_COLS]
    summary = (
        df.groupby(CLASS_AREA_GROUP_COLS, dropna=False, observed=True, as_index=False)["area_ha"]
          .sum()
          .sort_values(sort_cols)
          .reset_index(drop=True)
    )[CLASS_AREA_COLS]

    diff = (summary.groupby([cn.adm0_pattern, "year"])["area_ha"].sum() - input_totals).abs().max()
    log(f"IPCC class area: {len(summary):,} output rows; max country-year total difference vs input = {diff:,.6f} ha")

    return summary, rule_report


def write_workbook(tabs, path):
    """Writes each dataframe to its own tab: Arial, bold frozen header row, filters, number formats, column widths."""
    for name, df in tabs.items():
        if len(df) > EXCEL_MAX_ROWS:
            raise ValueError(f"{name} has {len(df):,} rows, more than Excel allows ({EXCEL_MAX_ROWS:,}). "
                             f"Write it to CSV or parquet instead, or keep fewer countries.")

    number_formats = {"value": "#,##0.00", "area_ha": "#,##0.0"}

    # xlsxwriter is much faster for large tabs; openpyxl is the fallback
    if importlib.util.find_spec("xlsxwriter"):
        with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
            book = writer.book
            book.formats[0].set_font_name("Arial")
            header_fmt = book.add_format({"bold": True, "font_name": "Arial", "bottom": 1, "text_wrap": True, "valign": "top"})
            for name, df in tabs.items():
                df.to_excel(writer, sheet_name=name, index=False)
                ws = writer.sheets[name]
                for i, col in enumerate(df.columns):
                    ws.write(0, i, col, header_fmt)
                    width = min(max(len(str(col)), int(df[col].astype(str).str.len().max() or 0)) + 2, 60)
                    fmt = book.add_format({"font_name": "Arial", "num_format": number_formats[col]}) if col in number_formats else None
                    ws.set_column(i, i, width, fmt)
                ws.freeze_panes(1, 0)
                ws.autofilter(0, 0, len(df), len(df.columns) - 1)
    else:
        from openpyxl.styles import Font
        from openpyxl.utils import get_column_letter
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for name, df in tabs.items():
                df.to_excel(writer, sheet_name=name, index=False)
                ws = writer.sheets[name]
                for i, col in enumerate(df.columns, start=1):
                    letter = get_column_letter(i)
                    width = min(max(len(str(col)), int(df[col].astype(str).str.len().max() or 0)) + 2, 60)
                    ws.column_dimensions[letter].width = width
                    for (cell,) in ws.iter_rows(min_col=i, max_col=i):
                        cell.font = Font(name="Arial", bold=(cell.row == 1))
                        if cell.row > 1 and col in number_formats:
                            cell.number_format = number_formats[col]
                ws.freeze_panes = "A2"
                ws.auto_filter.ref = ws.dimensions

    log(f"Saved {path}")

## Summarize and write the workbook

In [ ]:
# Combined files if the run was combined; otherwise the per-tile files are summed here.
flux_df, flux_source_desc = read_run(
    [flux_dir], combined_prefix="ipcc_lulucf_zonal_stats_", tile_prefix="ipcc_lulucf_zonal_stats_", exclude=("_wide_",))
class_area_df, class_area_source_desc = read_run(
    [flux_dir, class_area_dir], combined_prefix="ipcc_class_area_by_country_", tile_prefix="ipcc_class_area_")

flux_summary, flux_rule_report = summarize_flux(flux_df, flux_source_desc)
class_area, class_area_rule_report = summarize_class_area(class_area_df, class_area_source_desc)
del flux_df, class_area_df
gc.collect()

timestamp = uu.timestr()
country_label = "EU27" if FILTER_TO_EU27 else ("_".join(countries) if countries else "all_countries")
workbook_path = Path(output_dir) / f"IPCC_LULUCF_zonal_stats_summary__{zonal_stats_description}__{country_label}__{timestamp}.xlsx"

write_workbook({"flux": flux_summary, "ipcc_class_area": class_area}, workbook_path)
workbook_path

## QA

In [ ]:
# Rows (and, for the class area tab, area) changed by each reassignment rule.
# Flux area_ha is repeated for every analysis layer, so only the class area tab's area_ha is a real area.
display(pd.concat([flux_rule_report, class_area_rule_report], ignore_index=True))

In [ ]:
# Total IPCC class area per country and year (Mha), after reassignment. Compare with official country areas;
# GADM boundaries include inland water (IPCC class "Other Land (Water)").
display(
    class_area.assign(area_Mha=class_area["area_ha"] / 1_000_000)
              .pivot_table(index=["country_name", "year"], columns="IPCC_class_name", values="area_Mha", aggfunc="sum", fill_value=0)
              .assign(total_Mha=lambda d: d.sum(axis=1))
)

In [ ]:
# Consistency: the second digit of IPCC_change is the class at the end of the interval, so it should equal IPCC_class.
# Area where it doesn't (excluding change code 0, Unassigned):
check = class_area[class_area["IPCC_change"] != 0]
mismatch = check[(check["IPCC_change"] % 10) != check["IPCC_class"]]
print(f"Area where IPCC_change doesn't end in IPCC_class: {mismatch['area_ha'].sum():,.1f} ha of {check['area_ha'].sum():,.1f} ha")
display(mismatch.groupby(["IPCC_change_name", "IPCC_class_name"])["area_ha"].sum().sort_values(ascending=False).head(20))

In [ ]:
# Area by initial IPCC class and year (Mha)
display(
    class_area.assign(area_Mha=class_area["area_ha"] / 1_000_000)
              .pivot_table(index=["country_name", "IPCC_initial"], columns="year", values="area_Mha", aggfunc="sum", fill_value=0)
)

In [ ]:
# Flux totals by flux source and analysis layer
display(flux_summary.groupby(["flux_source", "analysis_layer"])[VALUE_COLS].sum())

In [ ]:
# Net flux by country and year for each flux source
net_layers = [l for l in flux_summary["analysis_layer"].unique() if "net" in l]
display(
    flux_summary[flux_summary["analysis_layer"].isin(net_layers)]
    .pivot_table(index=["country_name", "flux_source", "analysis_layer"], columns="year", values="value", aggfunc="sum")
)

In [ ]:
class_area.head(10)